In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine 

In [2]:
db_user = 'postgres'
db_password = 'p4pakistan' 
db_host = 'localhost'
db_port = '5432'
db_name = 'ecommerce_churn_db'

connection_string = f'postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
engine = create_engine(connection_string)

In [3]:
# 2. Load the customer_rfm table into a Pandas DataFrame
query = "SELECT * FROM customer_rfm;"
df_rfm = pd.read_sql(query, con=engine)

# 3. Inspect the first few rows
print(df_rfm.head())
print(f"Total customers loaded: {len(df_rfm)}")

  customer_id         country  recency  frequency  monetary
0       12346  United Kingdom    325.0          1  77183.60
1       12347         Iceland      2.0          7   4310.00
2       12348         Finland     75.0          4   1797.24
3       12349           Italy     18.0          1   1757.55
4       12350          Norway    310.0          1    334.40
Total customers loaded: 4339


In [4]:
# 1. Create the target variable: 1 if recency > 90 days (Churned), else 0
df_rfm['is_churned'] = np.where(df_rfm['recency'] > 90, 1, 0)

# 2. Check the distribution of churned vs active customers
churn_counts = df_rfm['is_churned'].value_counts(normalize=True) * 100
print("Churn Distribution (%):")
print(churn_counts)

# 3. Quick look at the updated DataFrame
print(df_rfm.head())

Churn Distribution (%):
is_churned
0    66.605209
1    33.394791
Name: proportion, dtype: float64
  customer_id         country  recency  frequency  monetary  is_churned
0       12346  United Kingdom    325.0          1  77183.60           1
1       12347         Iceland      2.0          7   4310.00           0
2       12348         Finland     75.0          4   1797.24           0
3       12349           Italy     18.0          1   1757.55           0
4       12350          Norway    310.0          1    334.40           1


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# 1. Define Features (X) and Target (y)
# We drop customer_id and country (or you could encode country, but let's keep it simple with RFM features first)
X = df_rfm[['recency', 'frequency', 'monetary']]
y = df_rfm['is_churned']

# 2. Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Scale features (crucial for Logistic Regression / Gradient Boosting optimization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Train a Logistic Regression Model (Baseline Classification Model)
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# 5. Evaluate the Model
y_pred = model.predict(X_test_scaled)

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy Score: 0.9907834101382489

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       578
           1       1.00      0.97      0.99       290

    accuracy                           0.99       868
   macro avg       0.99      0.99      0.99       868
weighted avg       0.99      0.99      0.99       868



In [6]:
import joblib

# 1. Save the trained model and scaler
joblib.dump(model, 'churn_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("Model and scaler successfully saved as .pkl files!")

Model and scaler successfully saved as .pkl files!
